# Research Question

## Can property characteristics such as room type, number of reviews, availability, and accommodation capacity accurately predict Airbnb listing prices?

## Introduction

### By enabling hosts to post properties anywhere in the globe, Airbnb has revolutionized the short-term rental market and created a dynamic, competitive pricing environment. Travelers wishing to make wise choices and hosts hoping to maximize income can both benefit from knowing what influences listing costs. This study investigates the possibility of accurately predicting Airbnb listing costs based on property attributes, including room type, number of reviews, availability, and accommodation capacity. The analysis uses dimensionality reduction and statistical modeling to determine which characteristics have the most impact on price variation.

### An Airbnb listings CSV file with variables including price, room_type, minimum_nights, number_of_reviews, reviews_per_month, availability_365, accommodates, bathrooms, and bedrooms served as the project's dataset. The dataset is ideal for Principal Component Analysis (PCA) and Multiple Linear Regression since it contains both numerical and categorical variables in addition to linked characteristics. In order to ascertain whether dimensionality reduction enhances predictive accuracy or interpretability, this investigation compares model performance before and after PCA.

## Supervised Learning

### The association between Airbnb listing prices and property attributes is modeled in this study using multiple linear regression. The baseline regression model is trained using the entire feature set following dataset cleaning, which includes handling missing values, encoding categorical variables, and transforming price values from texts to numeric notation. To find the most significant predictors, Lasso regression is used for feature selection. Principal Component Analysis (PCA) is used to minimize dimensionality and handle multicollinearity since a number of characteristics (such as bedrooms, bathrooms, and accommodations) are interrelated.

### Two models one with PCA-transformed components and the other with the original features are compared. In order to preserve at least 90% of the variance, the number of primary components is determined using the explained variance ratio. R² and RMSE are used to assess model performance. This comparison shows if PCA simplifies the model without sacrificing accuracy or enhances predictive performance.

## Conclusion and Future Direction

### The findings demonstrate that a number of property attributes, such as room type, number of reviews, and accommodation capacity, significantly influence the prediction of Airbnb listing prices. The dataset has significant predictive properties, as evidenced by the baseline regression model's good performance. The model performs similarly with fewer components after PCA is applied, indicating that dimensionality reduction successfully captures the underlying structure of the data without significantly losing information.

### Future studies could use additional variables, such as local crime rates, walkability rankings, or seasonal demand trends, to further boost forecast accuracy. To compare performance with linear regression, more sophisticated machine learning models like Random Forests, Gradient Boosting, or Neural Networks could be investigated. Lastly, different market categories within Airbnb listings could be identified using clustering techniques, providing additional insights into host behavior and pricing tactics.

## Refrences

### Inside Airbnb. (2024). Airbnb listings dataset. https://insideairbnb.com/get-the-data/ 

### chodimella. (2020). airbnb-listings.csv. GitHub Gist.https://gist.github.com/chodimella/60e9aa590635d2ec6ad88e28497b042c#file-airbnb-listings-csv 

## Bring in the libraries required for modeling, evaluation, and data processing.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

## Open a pandas DataFrame and load the Airbnb listings dataset.

In [2]:
df = pd.read_csv("Downloads/airbnb-listings.csv")
df.head()

,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,instant_bookable,is_business_travel_ready,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,10080,https://www.airbnb.com/rooms/10080,2.020000e+13,2019-02-06,D1 - Million Dollar View 2 BR,"Stunning two bedroom, two bathroom apartment. ...","Bed setup: 2 x queen, I can add up to 2 twin s...","Stunning two bedroom, two bathroom apartment. ...",none,NaN,...,f,f,strict_14_with_grace_period,f,f,31,31,0,0,0.18
1,11400,https://www.airbnb.com/rooms/11400,2.020000e+13,2019-02-06,Central Lovely Rm in Victorian Home,Well-appointed room with a view of the garden ...,"Centrally-located lovely, quiet home on tree-l...",Well-appointed room with a view of the garden ...,none,"Very quiet residential area, yet only 1-1/2 bl...",...,f,f,strict_14_with_grace_period,t,t,1,0,1,0,0.64
2,13188,https://www.airbnb.com/rooms/13188,2.020000e+13,2019-02-06,Garden level studio in ideal loc.,Garden level studio suite with garden patio - ...,"Explore Vancouver in a highly sought after, tr...",Garden level studio suite with garden patio - ...,none,NaN,...,t,f,moderate,f,f,1,1,0,0,1.51
3,13357,https://www.airbnb.com/rooms/13357,2.020000e+13,2019-02-06,! Wow! 2bed 2bath 1bed den Harbour View Apartm...,Very spacious and comfortable with very well k...,"Mountains and harbour view 2 bedroom,2 bath,1 ...",Very spacious and comfortable with very well k...,none,Amanzing bibrant professional neighbourhood. C...,...,f,f,strict_14_with_grace_period,t,t,2,2,0,0,0.51
4,13358,https://www.airbnb.com/rooms/13358,2.020000e+13,2019-02-06,Urban Boutique Suite heart of Downtown Vancouver!,NaN,Welcome to the Electra Building on Nelson stre...,Welcome to the Electra Building on Nelson stre...,none,NaN,...,f,f,strict_14_with_grace_period,f,t,1,1,0,0,3.65


## Eliminate symbols and convert data to numeric representation to clean up the pricing column.

In [3]:
df['price'] = (
    df['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

## To guarantee valid regression targets, exclude listings with zero or invalid pricing.

In [4]:
df = df[df['price'] > 0]

## Determine the goal variable and choose the pertinent predictor variables.

In [5]:
features = [
    'room_type', 'minimum_nights', 'number_of_reviews', 'reviews_per_month',
    'availability_365', 'accommodates', 'bathrooms', 'bedrooms'
]

X = df[features]
y = df['price']

## Create a preprocessing pipeline that encodes categorical variables, scales numerical features, and imputes missing values.

In [6]:
numeric_features = [
    'minimum_nights', 'number_of_reviews', 'reviews_per_month',
    'availability_365', 'accommodates', 'bathrooms', 'bedrooms'
]

categorical_features = ['room_type']

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),

        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first'))
        ]), categorical_features)
    ]
)

## To evaluate the model, divide the dataset into training and testing sets.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Use the entire feature set to train a baseline Multiple Linear Regression model.

In [8]:
baseline_model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('regressor', LinearRegression())
])

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)

baseline_r2 = r2_score(y_test, y_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

baseline_r2, baseline_rmse

(0.1534469867493793, np.float64(213.3396777589889))

## To pick features and find significant predictors, use Lasso regression.

In [9]:
lasso_model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('lasso', Lasso(alpha=0.1))
])

lasso_model.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('lasso', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Reduce dimensionality using PCA, then use the modified components to train a regression model.

In [10]:
pca = PCA(n_components=0.90)  

pca_model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('pca', pca),
    ('regressor', LinearRegression())
])

pca_model.fit(X_train, y_train)
y_pca_pred = pca_model.predict(X_test)

pca_r2 = r2_score(y_test, y_pca_pred)
pca_rmse = np.sqrt(mean_squared_error(y_test, y_pca_pred))

pca_r2, pca_rmse

(0.12459258579752919, np.float64(216.9450099032475))

In [ ]:
## 